a) Определить влияние возраста на содержание иммуноглобулина в крови с помощью регрессионного анализа

In [7]:
import numpy as np
from scipy import stats 

y = np.array([83, 85,
              84, 85, 85, 86, 86, 87,
              86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90,
              89, 90, 90, 91,
              90, 92])

groups = np.repeat([1, 2, 3, 4, 5], [2, 6, 11, 4, 2])
n = len(y)
p = 5
psi = np.zeros((n, p))
for i in range(n):
    psi[i, groups[i] - 1] = 1

F = psi.T @ psi
F_inv = np.linalg.inv(F)
beta = F_inv @ psi.T @ y

e = y - psi @ beta
RSS = e.T @ e
TSS = np.sum((y - np.mean(y)) ** 2)

R2 = (TSS - RSS) / TSS
print(R2)

0.8106060606060607


In [11]:
delta = ((TSS - RSS) / RSS)* ((n - p) / (p - 1))
p_value = 1 - stats.f.cdf(delta, p-1, n-p)

print(p_value)


5.407435041959729e-07


b) Провести попарное сравнение средних в рамках регрессионной модели, с учетом множественности проверяемых гипотез. 

In [40]:
alpha = 0.05
m = 10
p_value_arr = []
for i in range(p):
    for j in range(i+1, p):
        delta = (beta[i] - beta[j]) / np.sqrt(RSS * (F_inv[i][i] + F_inv[j][j])) * np.sqrt(n-p)
        pv = (2*(1 - stats.t.cdf(np.abs(delta), df=n-p)))
        p_value_arr.append([pv, i, j])

p_value_arr = np.array(p_value_arr)
p_value_arr = p_value_arr[p_value_arr[:, 0].argsort()]

for k in range(m):
    print(f"Группы {int(p_value_arr[k][1])+1} и {int(p_value_arr[k][2])+1}:")
    if p_value_arr[k][0] < alpha / (m - k):
        print(f"Отвергаем H0, {p_value_arr[k][0]} < {alpha / (m-k)}")
    else:
        print(f"Нет оснований отвергать H0, p-value = {p_value_arr[k][0]} >= {alpha / (m-k)}")
        break


Группы 1 и 5:
Отвергаем H0, 2.4125702768884594e-06 < 0.005
Группы 2 и 4:
Отвергаем H0, 2.5534724907849693e-06 < 0.005555555555555556
Группы 1 и 4:
Отвергаем H0, 2.782111500732043e-06 < 0.00625
Группы 2 и 5:
Отвергаем H0, 4.085183105129175e-06 < 0.0071428571428571435
Группы 1 и 3:
Отвергаем H0, 0.0001662014487755492 < 0.008333333333333333
Группы 2 и 3:
Отвергаем H0, 0.00039504010033408754 < 0.01
Группы 3 и 5:
Отвергаем H0, 0.0010025629060046448 < 0.0125
Группы 3 и 4:
Отвергаем H0, 0.0023933387043622023 < 0.016666666666666666
Группы 1 и 2:
Нет оснований отвергать H0, p-value = 0.10310040890082384 >= 0.025
